# V13 — train and score leave-one-station-out

This runs on Colab because of two hard limits on the laptop the data lives on:
VGG19 backpropagates at **2.7 images/s** there, so a sixteen-fold sweep is about
**113 hours**, and the head alone still needs **6 hours per fold**.

## What to upload first

Produce these two locally (about 20 minutes), then put them in Drive under
`MyDrive/primates_v13/`:

```bash
python scripts/build_v13_dataset.py     # -> data/outputs/v13_manifest.csv
python scripts/pack_v13_images.py       # -> data/outputs/v13_images.npy  (4.77 GB)
                                        #    data/outputs/v13_index.csv
```

Only those go up. The 444 GB of field recordings stay on the external drive —
nothing here needs them, because every clip has already been reduced to the exact
2 s analysis window the model sees, verified sample-for-sample against the source
recordings.

## What comes back

`v13_loso.csv`: per held-out station, how many of that station's reviewed false
positives the new model rejects and how many of its confirmed calls it keeps.
That is **precision**, measured out of sample. It is not recall — a call V12
never fired on was never exported and never reviewed, so no experiment on this
set can see it.

In [ ]:
!nvidia-smi -L
import tensorflow as tf
print("TF", tf.__version__, "GPUs:", tf.config.list_physical_devices('GPU'))

## 1. Repository and data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/primates_v13'
!ls -la $DRIVE

In [ ]:
# The repo. The V13 scripts live on v13-honest-labels, NOT on main -- a clone
# without -b lands on main, where train_v13_loso.py does not exist and the
# cells below fail with a bare FileNotFoundError.
!git clone -b v13-honest-labels https://github.com/mo119m/primates-sound-detection.git /content/repo
%cd /content/repo
!git rev-parse --abbrev-ref HEAD
!git pull --ff-only 2>/dev/null || true
!pip -q install librosa soundfile

In [ ]:
import os, shutil
os.makedirs('/content/repo/data/outputs', exist_ok=True)

# Copy off Drive to local disk first: training reads the feature cache every
# epoch, and Drive's FUSE mount makes that far slower than the GPU.
# v13_manifest.csv is NOT optional. The pack contains 657 clips from the mahal/
# and yamnet/ dumps that no human has ever labelled; the index carries them as
# Background because packing needed a value, and the manifest is the only thing
# that keeps them out of training. Omitting it here used to train on them
# silently -- train_v13_loso.py now refuses to start instead.
for name in ('v13_images.npy', 'v13_index.csv', 'v13_manifest.csv'):
    src, dst = f'{DRIVE}/{name}', f'/content/repo/data/outputs/{name}'
    if not os.path.exists(src):
        raise FileNotFoundError(
            f'{src} is missing. All three files must be on Drive: the pack, the '
            'index, and the manifest.')
    if not os.path.exists(dst):
        print('copying', name); shutil.copy(src, dst)
print(os.popen('ls -la /content/repo/data/outputs').read())

## 2. Feature cache

The VGG19 base is frozen for stage 1, so its output for a given image is the same
in every epoch and every fold. Computing it once and keeping the `block4_conv4`
activations is not an approximation of stage 1 — it *is* stage 1, with the
constant part evaluated once instead of sixteen times over.

About 25 GB at float16. Colab's local disk holds it; Drive should not be used for
this file.

In [ ]:
# One fold first, to see a number before committing to the sweep.
!python scripts/train_v13_loso.py --folds IPA20ST --epochs 15 --verbose 1

## 3. The full sweep

Sixteen folds. Each withholds one station from training entirely — including the
1 348 clips whose origin narrows to a group of stations rather than to one,
because five of the sixteen were deployed without position metadata and cannot
be told apart; guessing between them would leak quietly into every fold.

In [ ]:
!python scripts/train_v13_loso.py --folds all --epochs 15

In [ ]:
import pandas as pd
df = pd.read_csv('/content/repo/data/outputs/v13_loso.csv')
df

In [ ]:
c, f = df['calls'].sum(), df['false_positives'].sum()
kc, kf = df['kept_calls'].sum(), df['kept_false_positives'].sum()
print(f"V12 precision {c/(c+f):.3f}   ({c} calls / {c+f} detections)")
print(f"V13 precision {kc/(kc+kf):.3f}   ({kc} calls / {kc+kf} detections)")
print(f"calls retained {kc/c:.3f}   false positives removed {1-kf/f:.3f}")
print(f"review reduced by {1-(kc+kf)/(c+f):.3f}")

### Excluding IPA4ST

One station holds 2 370 of the 3 654 reviewed false positives — an untrained
species called there in long bouts. It dominates any pooled figure, and the
paper already reports the fifteen-station numbers alongside the sixteen, so both
belong here too.

In [ ]:
sub = df[df.station != 'IPA4ST']
c, f = sub['calls'].sum(), sub['false_positives'].sum()
kc, kf = sub['kept_calls'].sum(), sub['kept_false_positives'].sum()
print(f"15 stations, excluding IPA4ST")
print(f"V12 precision {c/(c+f):.3f}  ->  V13 precision {kc/(kc+kf):.3f}")
print(f"calls retained {kc/c:.3f}   false positives removed {1-kf/f:.3f}")

## 4. Stage 2 — unfreezing the last VGG blocks

Everything above trains the head on cached features, which is exactly stage 1.
Stage 2 fine-tunes blocks 4 and 5 at a low learning rate, and cannot use the
cache because those weights move. It needs the images, and it needs this GPU.

Run it only after stage 1 shows a gain worth extending, and read the fold numbers
before deciding: fine-tuning more parameters on the same labels can just as
easily fit the stations it was given.

In [ ]:
# Placeholder for the stage-2 sweep. Fill in once stage-1 folds are in:
# the schedule is the V12 one -- unfreeze block4/block5, Adam at 1e-5 --
# applied per fold with the same withheld set.
print('stage 1 first')

## 5. Save results back to Drive

In [ ]:
!cp /content/repo/data/outputs/v13_loso.csv $DRIVE/
print('saved')